# Inspect interactions of one retained 3PTB pose

**Run All replays the explicit preparation once and performs six interaction
analyses. It executes no docking search, scoring, optimization or viewer.**
The original five-pose result is read unchanged; only its first stored pose is
selected. A separate +60 angstrom translation is a geometry control, not another
native pose. Two declared water geometries independently control hydrogen bonds.

Use the five preparation inputs in [the preparation guide](installed_3ptb_preparation.md),
plus `DOCKINGMT_INTERACTIONS_PREPARATION_NOTEBOOK` and its `_SHA256`,
`DOCKINGMT_INTERACTIONS_RESULT` and its `_SHA256`, and a fresh
`DOCKINGMT_INTERACTIONS_OUTPUT`. Notebook identities authorize executed code.
Use an ordinary-installed IPython/Jupyter kernel with the qualified providers.
Molecular assembly, H materialization, detection, queries and persistence are
public MolSysMT operations. The [registration](installed_3ptb_interactions_registration.md)
keeps this finite workflow separate from scientific, public and live-visual acceptance.

A registration and append-only progress log are written before preparation.
A started event without completion remains unknown after interruption; it is
never an empty scientific result. Partial output directories are preserved.


In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
import shlex
import sys
from copy import deepcopy
from pathlib import Path

import molsysmt as msm
import numpy as np
import pyunitwizard as puw
from IPython import get_ipython

import dockingmt as dmt

interaction_output = Path(os.environ['DOCKINGMT_INTERACTIONS_OUTPUT']).resolve()
interaction_result_path = Path(os.environ['DOCKINGMT_INTERACTIONS_RESULT']).resolve()
interaction_preparation_path = Path(
    os.environ['DOCKINGMT_INTERACTIONS_PREPARATION_NOTEBOOK']
).resolve()
interaction_inputs = {}
for name, path, variable in (
    (
        'preparation',
        interaction_preparation_path,
        'DOCKINGMT_INTERACTIONS_PREPARATION_NOTEBOOK_SHA256',
    ),
    ('result', interaction_result_path, 'DOCKINGMT_INTERACTIONS_RESULT_SHA256'),
):
    expected = os.environ[variable].lower()
    if len(expected) != 64 or any(c not in '0123456789abcdef' for c in expected):
        raise ValueError('Supply externally expected identities.')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('Input differs from its externally expected identity.')
    interaction_inputs[name] = expected
if interaction_output.exists():
    raise FileExistsError('Use a fresh interactions output directory.')
for module in (dmt, msm, puw):
    assert Path(module.__file__).resolve().is_relative_to(Path(sys.prefix).resolve())
assert (
    not json.loads(
        metadata.distribution('dockingmt').read_text('direct_url.json') or '{}'
    )
    .get('dir_info', {})
    .get('editable', False)
)
assert get_ipython() is not None
interaction_output.mkdir(parents=True, exist_ok=False)
interaction_progress_path = interaction_output / 'progress.jsonl'
interaction_progress_path.touch(exist_ok=False)


def retain_interaction_event(event):
    with interaction_progress_path.open('a') as stream:
        stream.write(json.dumps(event, allow_nan=False) + '\n')
        stream.flush()
        os.fsync(stream.fileno())


with (interaction_output / 'registration.json').open('x') as stream:
    json.dump(
        {
            'scope': 'one retained pose and separate geometry/analytic controls',
            'inputs': interaction_inputs,
            'selected_original_position': 1,
            'planned_detector_calls': 6,
            'new_searches_or_energy_calls': 0,
            'retry_policy': 'none; retain partial outcomes and register amendments',
            'reader_versions': {
                name: metadata.version(name)
                for name in ('dockingmt', 'molsysmt', 'pyunitwizard', 'rdkit')
            },
        },
        stream,
        allow_nan=False,
        indent=2,
    )
retain_interaction_event({'stage': 'workflow', 'status': 'started'})
print({'scope': 'six interaction analyses; no search/scoring/viewer'})

## Reuse the preparation and verify the original pose axis

The companion retains its own preparation-only receipt and original chemical
hypothesis: dry HID57/HIE40/HIE91, CYX links, charged termini and benzamidinium +1.
It verifies the source and supplied maps, creates H through MolSysMT and checks
all original prepared controls. This notebook neither discovers a correspondence
nor repairs PDBQT chemistry. It requires the declared thirteen-atom identity axis.

In [ ]:
retain_interaction_event({'stage': 'preparation', 'status': 'started'})
get_ipython().run_line_magic('run', shlex.quote(str(interaction_preparation_path)))
interaction_preparation = get_ipython().user_ns
interaction_preparation_receipt = deepcopy(interaction_preparation['receipt'])
interaction_prepared_partner = interaction_preparation['prepared_partner']
interaction_receptor = interaction_preparation['receptor_h']
interaction_prepared_before = deepcopy(interaction_prepared_partner.to_dict())
interaction_receptor_coordinates = msm.get(
    interaction_receptor, coordinates=True
).copy()
retain_interaction_event({'stage': 'preparation', 'status': 'completed'})
interaction_caller_policy = puw.configure.report()
interaction_raw = interaction_result_path.read_bytes()
if interaction_result_path.suffix == '.gz':
    interaction_raw = gzip.decompress(interaction_raw)
interaction_record = json.loads(interaction_raw)
assert dmt.audit_result(interaction_record)['status'] == 'consistent'
dmt.verify_captured_inputs(interaction_record['provenance']['backend_artifacts'])
interaction_record_before = deepcopy(interaction_record)
interaction_result = dmt.DockingResult.from_dict(interaction_record)
assert len(interaction_result) == 5
interaction_pose = interaction_result.poses[0]
assert interaction_pose.n_atoms == 13
assert interaction_pose.partner_state_id == interaction_prepared_partner.state_id
assert (
    interaction_pose.receptor_state_id
    == interaction_preparation['prepared_receptor'].state_id
)
assert interaction_pose.metadata['pose_atom_order'] == 'verified_pdbqt_order'
assert interaction_pose.metadata['selected_atom_indices'] == list(range(13))
assert interaction_pose.metadata['prepared_atom_indices'] == list(range(13))
interaction_ligand = msm.copy(interaction_prepared_partner.source_molsys)
interaction_fields = {
    'atom_id': 'atom_id',
    'atom_name': 'atom_name',
    'element': 'atom_type',
    'group_id': 'group_id',
    'group_name': 'group_name',
}
for key, attribute in interaction_fields.items():
    actual = [
        str(x) for x in msm.get(interaction_ligand, element='atom', **{attribute: True})
    ]
    assert actual == [x[key] for x in interaction_pose.metadata['source_atom_keys']]
msm.set(
    interaction_ligand,
    coordinates=msm.get(interaction_pose.coordinates, coordinates=True),
)
np.testing.assert_allclose(
    puw.get_value(msm.get(interaction_ligand, coordinates=True), to_unit='angstrom')[0],
    puw.get_value(interaction_pose.coordinates, to_unit='angstrom'),
    rtol=0,
    atol=1e-12,
)
print({'selected_original_position': 1, 'original_poses': 5, 'verified_pose_atoms': 13})

## Materialize omitted H through the fixed-state provider

The prepared native source retains the virtual H inventory from MolSysMT's
nonpolar-H projection. Ask the fixed-state builder to materialize it on the
actual pose, preserving all thirteen existing coordinates and polar H. The five
new nonpolar H are locally generated coordinates, not returned docking atoms or
environmentally refined positions. Preparation suitability remains unassessed.

In [ ]:
retain_interaction_event({'stage': 'pose_H_materialization', 'status': 'started'})
interaction_hydrogens = msm.build.add_missing_hydrogens(
    interaction_ligand,
    mode='fixed_chemical_state',
    pH=None,
    engine='RDKit',
    return_report=True,
    attribute_policy='intersection',
)
interaction_complete_ligand = interaction_hydrogens['molecular_system']
interaction_h_report = interaction_hydrogens['report']
assert interaction_h_report['n_added_hydrogens'] == 5
assert interaction_h_report['original_coordinates_preserved']
assert msm.get(interaction_complete_ligand, n_atoms=True) == 18
interaction_h_map = np.asarray(interaction_h_report['atom_correspondence'], dtype=int)
np.testing.assert_array_equal(interaction_h_map[:, 0], np.arange(13))
np.testing.assert_allclose(
    puw.get_value(
        msm.get(interaction_complete_ligand, coordinates=True), to_unit='angstrom'
    )[0, interaction_h_map[:, 1]],
    puw.get_value(interaction_pose.coordinates, to_unit='angstrom'),
    rtol=0,
    atol=1e-12,
)
retain_interaction_event(
    {'stage': 'pose_H_materialization', 'status': 'completed', 'new_H': 5}
)
retain_interaction_event({'stage': 'assembly', 'status': 'started'})
interaction_models = {
    'retained_pose': msm.merge(
        [interaction_receptor, interaction_complete_ligand], keep_ids=True
    ),
    'translated_geometry_control': msm.merge(
        [
            interaction_receptor,
            msm.structure.translate(
                interaction_complete_ligand,
                translation=puw.quantity([60, 0, 0], 'angstrom'),
                in_place=False,
            ),
        ],
        keep_ids=True,
    ),
}
interaction_scopes = {}
for name, model in interaction_models.items():
    receptor_atoms = msm.select(model, selection="molecule_type=='protein'")
    ligand_atoms = msm.select(model, selection="group_name=='BEN'")
    assert len(ligand_atoms) == 18
    assert len(receptor_atoms) == msm.get(interaction_receptor, n_atoms=True)
    assert set(receptor_atoms).isdisjoint(ligand_atoms)
    assert len(receptor_atoms) + len(ligand_atoms) == msm.get(model, n_atoms=True)
    interaction_scopes[name] = (receptor_atoms, ligand_atoms)
retain_interaction_event({'stage': 'assembly', 'status': 'completed'})
print({'materialized_H': 5, 'cases': list(interaction_models), 'pbc': False})

## Detect named families and preserve complete typed analyses

The two MolSysMT criteria retain their exact cutoff inequalities and recognition
profiles. Querying and persistence do not credit new calculations. H5MSM stores
the molecular context and attached analyses; typed `InteractionsDict` is the
complete interchange. Inspection dictionaries/pages do not replace that codec.

In [ ]:
interaction_counts = {}
interaction_analyses = {}
interaction_files = {}
for name, model in interaction_models.items():
    receptor_atoms, ligand_atoms = interaction_scopes[name]
    options = dict(
        selection=ligand_atoms,
        selection_2=receptor_atoms,
        selection_mode='between',
        structure_indices=[0],
        chemical_state='reference',
        pbc=False,
        assume_complete_connectivity=False,
    )
    analyses = {}
    families = {
        'hbonds': (
            msm.interactions.hbonds.get_hbonds,
            'baker_hubbard',
            'nitrogen_oxygen',
        ),
        'hydrophobic': (
            msm.interactions.hydrophobic.get_hydrophobic_interactions,
            'atom_pair_distance',
            'smarts_hydrophobic_atoms',
        ),
    }
    for family, (detector, method, profile) in families.items():
        retain_interaction_event(
            {'stage': 'detector', 'case': name, 'family': family, 'status': 'started'}
        )
        try:
            analyses[family] = detector(
                model, method=method, profile=profile, **options
            )
        except Exception as error:
            retain_interaction_event(
                {
                    'stage': 'detector',
                    'case': name,
                    'family': family,
                    'status': 'failed',
                    'exception': type(error).__name__,
                    'code': getattr(error, 'code', None),
                }
            )
            raise
        retain_interaction_event(
            {
                'stage': 'detector',
                'case': name,
                'family': family,
                'status': 'completed',
                'observations': int(analyses[family].n_interactions),
            }
        )
    model.interactions = analyses
    filename = name + '.h5msm'
    path = interaction_output / filename
    assert not path.exists()
    retain_interaction_event(
        {'stage': 'provider_persistence', 'case': name, 'status': 'started'}
    )
    msm.convert(model, to_form='file:h5msm', output_filename=path)
    restored = msm.convert(path, to_form='molsysmt.MolSys')
    interaction_counts[name] = {}
    for family, analysis in analyses.items():
        expected = msm.convert(analysis, to_form='molsysmt.InteractionsDict').to_dict()
        typed_restored = msm.convert(
            msm.convert(analysis, to_form='molsysmt.InteractionsDict'),
            to_form='molsysmt.Interactions',
        )
        for received in (typed_restored, restored.interactions[family]):
            np.testing.assert_equal(
                expected,
                msm.convert(received, to_form='molsysmt.InteractionsDict').to_dict(),
            )
            np.testing.assert_array_equal(received.evaluated_structure_indices, [0])
            filtered = received.between_selections(
                ligand_atoms, receptor_atoms, exclusive=True
            )
            assert filtered.n_interactions == received.n_interactions
        interaction_counts[name][family] = analysis.n_interactions
    retain_interaction_event(
        {'stage': 'provider_roundtrip', 'case': name, 'status': 'completed'}
    )
    interaction_analyses[name] = analyses
    interaction_files[filename] = {
        'bytes': path.stat().st_size,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    }
assert all(
    count == 0 for count in interaction_counts['translated_geometry_control'].values()
)
print({'observations': interaction_counts, 'full_typed_roundtrips': True})

## Independent analytic hydrogen-bond controls

Build one declared water using public MolSysMT conversion and fixed-state H
addition, then merge two copies. The supplied six-atom coordinates are an analytic
software fixture, not a prepared biological complex. Its positive donor-H-acceptor
line has H-A = 0.18 nm and D-H-A = 180 degrees. Translate the second water by
0.4 nm along y through MolSysMT for the adverse geometry. Both are evaluated;
empty observations remain different from an unevaluated result.

In [ ]:
retain_interaction_event({'stage': 'analytic_geometry', 'status': 'started'})
interaction_water = msm.convert('smiles:O', to_form='molsysmt.MolSys')
assert msm.get(interaction_water, n_structures=True) == 0
interaction_water.structures.append(coordinates=puw.quantity([[[0, 0, 0]]], 'nm'))
interaction_water = msm.build.add_missing_hydrogens(
    interaction_water,
    mode='fixed_chemical_state',
    pH=None,
    engine='RDKit',
)
assert list(msm.get(interaction_water, element='atom', atom_type=True)) == [
    'O',
    'H',
    'H',
]
interaction_toy = msm.merge([interaction_water, msm.copy(interaction_water)])
msm.set(
    interaction_toy,
    coordinates=puw.quantity(
        [
            [
                [0, 0, 0],
                [0.1, 0, 0],
                [0, 0.1, 0],
                [0.28, 0, 0],
                [0.28, 0.1, 0],
                [0.28, 0, 0.1],
            ]
        ],
        'nm',
    ),
)
interaction_toy_models = {
    'analytic_positive': interaction_toy,
    'analytic_adverse': msm.structure.translate(
        interaction_toy,
        selection=[3, 4, 5],
        translation=puw.quantity([0, 0.4, 0], 'nm'),
        in_place=False,
    ),
}
interaction_toy_counts = {}
for name, model in interaction_toy_models.items():
    retain_interaction_event(
        {'stage': 'detector', 'case': name, 'family': 'hbonds', 'status': 'started'}
    )
    analysis = msm.interactions.hbonds.get_hbonds(
        model,
        selection=[0, 1, 2],
        selection_2=[3, 4, 5],
        selection_mode='between',
        structure_indices=[0],
        method='baker_hubbard',
        profile='nitrogen_oxygen',
        pbc=False,
    )
    retain_interaction_event(
        {
            'stage': 'detector',
            'case': name,
            'family': 'hbonds',
            'status': 'completed',
            'observations': int(analysis.n_interactions),
        }
    )
    interaction_toy_counts[name] = analysis.n_interactions
    np.testing.assert_array_equal(analysis.evaluated_structure_indices, [0])
    if name == 'analytic_positive':
        assert analysis.n_interactions == 1
        np.testing.assert_allclose(
            analysis.measurements['hydrogen_acceptor_distance'],
            0.18,
            rtol=0,
            atol=1e-12,
        )
        np.testing.assert_allclose(
            analysis.measurements['dha_angle'], np.pi, rtol=0, atol=1e-12
        )
    else:
        assert analysis.n_interactions == 0
    model.interactions = {'hbonds': analysis}
    filename = name + '.h5msm'
    path = interaction_output / filename
    msm.convert(model, to_form='file:h5msm', output_filename=path)
    interaction_files[filename] = {
        'bytes': path.stat().st_size,
        'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
    }
print({'analytic_controls': interaction_toy_counts})

## Save docking association separately from molecular persistence

The manifest binds each provider file to this exact original result and selected
pose. The translated and analytic controls are explicitly separate populations.
The source result and scores remain unchanged. Reading the files later does not
recompute interactions, generate H or execute docking. The addon can consume the
provider's structured analyses; no actual canvas acceptance is claimed here.

In [ ]:
assert interaction_record == interaction_record_before
assert (
    hashlib.sha256(interaction_result_path.read_bytes()).hexdigest()
    == interaction_inputs['result']
)
np.testing.assert_equal(
    interaction_prepared_partner.to_dict(), interaction_prepared_before
)
np.testing.assert_allclose(
    puw.get_value(msm.get(interaction_receptor, coordinates=True), to_unit='angstrom'),
    puw.get_value(interaction_receptor_coordinates, to_unit='angstrom'),
    rtol=0,
    atol=1e-12,
)
assert puw.configure.report() == interaction_caller_policy
interaction_association = {
    'schema': 'dockingmt.3ptb_interaction_association@1',
    'scope': 'one retained pose, separate translated geometry and analytic controls',
    'inputs': interaction_inputs,
    'original_decoded_result_sha256': hashlib.sha256(interaction_raw).hexdigest(),
    'original_producer': {
        k: interaction_record['provenance'][k]
        for k in (
            'dockingmt_version',
            'molsysmt_version',
            'backend',
            'backend_version',
        )
    },
    'original_pose_count': 5,
    'selected_original_position': 1,
    'pose_id': interaction_pose.pose_id,
    'partner_state_id': interaction_pose.partner_state_id,
    'receptor_state_id': interaction_pose.receptor_state_id,
    'original_scores': interaction_pose.scores,
    'pose_coordinates_sha256': hashlib.sha256(
        json.dumps(
            interaction_record['poses'][0]['coordinates'],
            sort_keys=True,
            separators=(',', ':'),
            allow_nan=False,
        ).encode()
    ).hexdigest(),
    'pose_atom_keys': deepcopy(interaction_pose.metadata['source_atom_keys']),
    'existing_ligand_atom_correspondence': interaction_h_map.tolist(),
    'merged_ligand_atom_indices': {
        name: [int(index) for index in scopes[1]]
        for name, scopes in interaction_scopes.items()
    },
    'actual_pose_atoms_preserved': 13,
    'materialized_nonpolar_H': 5,
    'H_coordinates': 'MolSysMT fixed-state local placement; not environment refinement',
    'preparation_receipt': interaction_preparation_receipt,
    'analyses': interaction_counts,
    'analytic_controls': interaction_toy_counts,
    'files': interaction_files,
    'reader_versions': {
        n: metadata.version(n)
        for n in ('dockingmt', 'molsysmt', 'pyunitwizard', 'rdkit')
    },
    'progress_log': 'progress.jsonl',
    'interaction_detector_calls': 6,
    'new_searches_or_energy_calls': 0,
    'original_record_preserved': True,
    'caller_unit_policy_preserved': True,
    'chemical_model_suitability': 'unassessed',
    'public_package_or_scientific_panel_or_live_visual_acceptance': False,
}
with (interaction_output / 'association.json').open('x') as stream:
    json.dump(interaction_association, stream, allow_nan=False, indent=2)
retain_interaction_event({'stage': 'workflow', 'status': 'completed'})
print(
    {
        'saved_provider_files': list(interaction_files),
        'interaction_calls': 6,
        'new_searches': 0,
    }
)